### Pydantic 모델
- 정의된 데이터만 전송되도록 요청 바디 검증

In [1]:
from datetime import datetime

from pydantic import BaseModel, PositiveInt
from pydantic import ValidationError


class User(BaseModel):
    id: int  
    name: str = 'John Doe'  
    signup_ts: datetime | None  
    tastes: dict[str, PositiveInt]  


external_data = {
    'id': 123,
    'signup_ts': '2019-06-01 12:22',  
    'tastes': {
        'wine': 9,
        b'cheese': 7,  
        'cabbage': '1',  
    },
}

# user = User(id=1234,signup_ts = '2019-06-01 12:22')
user = User(**external_data)  

print(user.id)  
#> 123
print(user.model_dump())  # 모델의 필드와 값을 dict 형태로 반환


123
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [2]:
from pydantic import ValidationError

external_data = {'id':'not an int', 'tastes' : {}}

try:
    User(**external_data)
except ValidationError as e:
    print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [4]:
# Fruit: name - 문자열, color - red or green, 
# weight - 0보다 크고 float, bazam-dict[키-str, [tuple(int, bool, float)]]

from typing import Annotated, Literal
from annotated_types import Gt


# Gt(greater then)


class Fruit(BaseModel):
    name: str
    color: Literal["red","green"]
    weight: Annotated[float, Gt(0)]
    bazam: dict[str, list[tuple[int, bool, float]]]

In [5]:
fruit = Fruit(name='Apple', color='red', weight=4.2, bazam={'foobar':[(1,True,0.1)]})

In [6]:
fruit = Fruit(name='Apple', color = 'red', weight=4.2)
fruit

ValidationError: 1 validation error for Fruit
bazam
  Field required [type=missing, input_value={'name': 'Apple', 'color': 'red', 'weight': 4.2}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [7]:
# Book - code(4자리), category(소설, 과학, 역사, 경제, 자기계발, 컴퓨터), title-문자, writer-문자, price-int, 0보다 크다

from pydantic import StringConstraints
from annotated_types import Gt


class Book(BaseModel):
    code:Annotated[str, StringConstraints(min_length=4, max_length=4)]
    category: Literal["소설", "과학", "역사","경제", "자기계발", "컴퓨터"]
    title:str
    writer:str
    price: Annotated[int, Gt(0)]




In [ ]:
try:
    Book(code='1001', category='경제', price=25000)
except ValidationError as e:
    print(e.errors())

In [14]:
# User-id:int, name:문자, age 14~100, email,role='user' or 'admin'
# ge, le

from pydantic import Field

class User(BaseModel):
    id:int
    name:str
    age:Annotated[int,Field(ge=14, le=100)]
    email:str
    role:Literal['user', 'admin']


In [15]:
from pydantic import ValidationError
try:
    User(id=1, name='홍길동', age=101, email='hong@naver.com', role='user')
except ValidationError as e:
    print(e.errors())

[{'type': 'less_than_equal', 'loc': ('age',), 'msg': 'Input should be less than or equal to 100', 'input': 101, 'ctx': {'le': 100}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}]


In [18]:
# Product : code-문자 , name-문자, category-food, clothes, electronics, price-정수, 0보다 커야한다
# stock-,정수, 0이상


from annotated_types import Ge, Le

class Product(BaseModel):
    code:str
    name:str
    category:Literal["food", "clothes" , "electronics"]
    price: Annotated[int,Gt(0)]
    stock: Annotated[int,Ge(0)]



In [19]:
class Item(BaseModel):
    item:str
    status:str

class Cart(BaseModel):
    id:int
    item:Item

In [20]:
try:
    cart = Cart(id=1, item=Item(item='바지', status='판매'))

except ValueError as e:
    print(e.errors())

In [21]:
class OrderItem(BaseModel):
    product_name:str
    quantity:Annotated[int,Gt(0)]
    price:Annotated[int,Gt(0)]

class Order(BaseModel):
    order_id:int
    customer:str
    items:list[OrderItem]


In [22]:
order = Order(order_id=1001, customer='홍길동', items=[
    {"product_name" : "노트북", "quantity":1, "price":120000},
    {"product_name" : "마우스", "quantity":1, "price":120000},
])

In [23]:
# Comment : writer-문자, content-문자
# Post : id-숫자, title-문자, writer-문자, content-문자, comments: Comment 가 여러개
class Comment(BaseModel):
    writer:str
    content:str

class Post(BaseModel):
    id:int
    title:str
    writer:str
    content:str
    comments: list[Comment]

In [24]:
comment = Comment(id=1, title='안녕하세요', writer='이종혁', content='안녕', comments=[
    {"writer" : "나", "content":"ㅎㅇ"}
])